In [ ]:
import numpy as np 
from zomba.singleObjective.simulator import SLBSimulator
from zomba.singleObjective.stochastic import SLBUCB

In [ ]:
K = 10
d = 5

env = SLBSimulator(num_arm=K, num_dim=d, vary_context=1)

In [ ]:
env.reset(verbose=1)

In [ ]:
alg = SLBUCB(num_arm=K, num_dim=d, rarely_switch=0, alpha=.5) 

In [ ]:
alg.reset()

In [ ]:
env.observe_context()

In [ ]:
a_t = alg.take_action(context=env.observe_context())

In [ ]:
env.get_reward(a_t)

In [ ]:
env.get_regret(a_t)

In [ ]:
T = 1000
alg.reset()
env.reset()
reg = []
for t in range(T): 
    X = env.observe_context() 
    a_t = alg.take_action(X) 

    r = env.get_reward(arm=a_t) 
    alg.update((a_t, r, X[a_t]))
    
    reg.append(env.get_regret(arm=a_t))
    if (t+1) % 100 == 0: 
        print(f"round: {t+1}, cumulative regret: {np.sum(reg)}")
    

In [ ]:
import matplotlib.pyplot as plt 

plt.plot(range(T+1), np.cumsum(np.insert(reg, 0, 0)) )
plt.show()

In [ ]:
class ucb_gpt(SLBUCB): 
    def update(self, info) -> None:
        super().update(info) 
        self.alpha = np.sqrt(np.log(self.round + 1) / (self.reward_his.size + round + 1 + 1e-8))



In [13]:
banditAlgs = {'UCB(0.5)': SLBUCB(num_arm=K, num_dim=d, alpha=.5), 
              'UCB(0.1)': SLBUCB(num_arm=K, num_dim=d, alpha=.1),
              'UCB(0.05)': SLBUCB(num_arm=K, num_dim=d, alpha=.05),
              'UCB(0.01)': SLBUCB(num_arm=K, num_dim=d, alpha=.01),
              'UCB(gpt)': ucb_gpt(num_arm=K, num_dim=d),}
from zomba.singleObjective import experimenter, plot_regrets
regrets = experimenter(banditAlgrithms=banditAlgs, environment=env, num_arm=K, num_rep=5, num_round=T)

In [ ]:

fig, ax = plot_regrets(regrets, True)